# Notebook 14: Frozen-method three-seed replication

This notebook completes the **same six action controllers** for **DSADS folds 0–3 and HAR fold 0, seeds 2027 and 2028**. It imports all five seed-2026 configurations from the completed Notebook 13. The final study has **15 configurations**: DSADS four folds × three seeds and HAR one official test split × three seeds. The primary family remains `source_mean_f1`; all 28 reported methods, including the diagnostic oracle, are retained. No new architecture, features, utility target, validation grid or harm guard is introduced.

## Run the single executable cell

1. Select **Runtime → Change runtime type → T4 GPU**, then run the code cell. The GPU stage trains **42 auxiliary classifier/reconstructor pairs**, using the existing fixed 30/25-epoch recipe, and saves clean/development-fault action caches. The existing main classifiers, reconstructors and baseline test caches are reused.
2. When the output says **GPU STAGE COMPLETE**, switch to **CPU** and rerun the **same cell**. It fits **150 small controller heads**, selects each configuration's settings from its own validation participants, freezes all ten new configurations, then evaluates the saved test candidates.
3. If the notebook pauses at its 75-minute session budget, rerun the same cell in the same runtime stage. Completed epochs, cache conditions, head fits and phases are reused. After a connection interruption, an unfinished epoch or unsaved head may need to run again. `RUN_MODE='all'` can execute both stages in one T4 session, but uses GPU quota during CPU fitting.

No manual uploads or dataset downloads are required. Keep the existing `processed/v1` and Notebook 02/04/05/09/11/12/13 folders in `MyDrive/SensorRepair_ICDSIS`. Notebook 13 must be complete. The notebook checks input hashes and the recorded scientific recipe. Neural fitting requires the PyTorch build recorded by Notebook 11; the CPU stage permits the CPU build. It checks the recorded scikit-learn version before loading/fitting controllers. Existing output folders are read without modification.

## What the replication measures

Each additional configuration has its **own source-only auxiliary models and controller fits**. Source cohorts are excluded from auxiliary fitting and normalization. Auxiliary checkpoints use fixed final epochs. Original policy-cohort predictions come from the original frozen model as before. Validation selects settings independently for each fold/seed with the same grid and empirical guards. All ten new validation selections are frozen before any new test prediction array is opened. The source stage reads the prepared whole-dataset container to select training rows and audit participant separation; it does not fit on validation/test rows.

Seeds change the original cached signal-model/corruption runs and the auxiliary initialization, augmentation, corruption and cohort order. The existing controller-head randomization and source-jackknife partition seed remain fixed at 2026; matched window subsets follow the unchanged job-specific rule. The three runs are **not** three independent sets of test participants. DSADS still has **eight unique test participants**, and HAR still has **nine**.

Primary aggregation: average condition Macro-F1 within each fold, average required folds equally within a seed, then average **complete seeds** equally. Partial folds are saved in `per_seed_summary` and clearly labeled; incomplete seeds are excluded from the overall aggregate. Imported seed-2026 scores must reproduce Notebook 13 exactly. Prediction probabilities from different corruption seeds are never averaged together. Standard deviations across three seed means are descriptive; they are not confidence intervals.

## Interpretation and outputs

This is an exploratory replication after earlier test inspection. Stuck faults are excluded from controller fitting, but previous stuck-fault outcomes have already been inspected. Test harm can exceed validation guards. HAR may continue to select keep-only. Source jackknife excludes controller outcome rows, rather than refitting the entire auxiliary pipeline independently for each jackknife block. The notebook makes no significance, population-risk, novelty or improvement guarantee.

Outputs: `SensorRepair_ICDSIS/outputs/notebook14_v1`. In addition to the aggregate, it saves per-seed/per-fold/per-condition/per-participant results, coverage, matched-control and source audits, saved decisions, and the frozen analysis plan. All runs, including negative results and keep-only configurations, are retained.

After **NOTEBOOK 14 COMPLETE**, share the final printed output and:

- `notebook14_aggregate_summary.csv`
- `notebook14_comparisons.csv`
- `notebook14_per_seed_summary.csv`
- `notebook14_harm_audit.csv`
- `notebook14_run_report.json`


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import gc
import hashlib
import json
import math
import random
import shutil
import time
import zlib
import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT=PROJECT/'outputs'/'notebook14_v1'
LOCAL=Path('/content/sensorrepair_notebook14_v1')
CODE_VERSION='sensorrepair_nb14_v1'
CELL_SIGNATURE='c8bc82169b883449973f6951c8f6f517243f8a167477c3bb546fc3251f5ef108'
RUN_MODE='auto'  # T4: auxiliary caches only. CPU: controllers and cached test evaluation.
SESSION_MINUTES=75  # Rerun the same cell to resume after a budget pause.
REQUIRE_GPU=True
BATCH_SIZE=128
INFERENCE_BATCH_SIZE=128
LEARNING_RATE=.001
SEEDS=[2026,2027,2028]
BASE_JOBS=[{'dataset':'dsads','fold':f'fold_{f}','seed':2026} for f in range(4)]
BASE_JOBS += [{'dataset':'har','fold':'fold_0','seed':2026}]
JOBS=[]
for seed in SEEDS[1:]:
    JOBS += [{'dataset':'dsads','fold':f'fold_{f}','seed':seed} for f in range(4)]
    JOBS += [{'dataset':'har','fold':'fold_0','seed':seed}]
DEV_CONDITIONS=['clean']+[f'{f}_{i}' for f in ['dropout','bias','gain'] for i in [1,2,3]]
CONDITIONS=DEV_CONDITIONS+[f'stuck_{i}' for i in [1,2,3]]
SCOPES={'clean':['clean'],'known_faults':DEV_CONDITIONS[1:],'stuck_excluded_from_fitting':CONDITIONS[10:]}
ORIGINAL_FAMILIES=['oof_accuracy_utility','oof_f1_utility','source_mean_f1','source_min_f1']
CONTROL_FAMILIES=['window_mean_f1','window_min_f1']
NEW_METHODS=ORIGINAL_FAMILIES+CONTROL_FAMILIES
OLD_METHODS=['keep','mask_suspect','reconstruct_suspect','residual_gate_mask','residual_gate_reconstruct',
 'confidence_select','oracle_diagnostic','harm_aware_policy','benefit_only_policy','confidence_margin_tuned',
 'residual_mask_tuned','residual_reconstruct_tuned','ternary_outcome_selector','utility_regression_selector',
 'uniform_fusion_safe','uniform_fusion_accuracy','global_fusion_safe','global_fusion_accuracy',
 'gated_fusion_safe','gated_fusion_accuracy','residual_stacking_safe','residual_stacking_accuracy']
METHODS=OLD_METHODS+NEW_METHODS
PRIMARY='source_mean_f1'
METRICS=['macro_f1','accuracy','decision_change_rate','helpful_rate','harmful_rate']
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert RUN_MODE in ['auto','all']
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True)
if DEVICE.type=='cuda':torch.backends.cuda.matmul.allow_tf32=False
SESSION_DEADLINE=time.monotonic()+SESSION_MINUTES*60
print('NOTEBOOK 14 | frozen-method three-seed replication | one executable cell',flush=True)
print('Imported: all five seed-2026 configurations. New: ten configurations, seeds 2027 and 2028.',flush=True)
print('Device:',torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU','| mode:',RUN_MODE,flush=True)
print('42 auxiliary classifier/reconstructor pairs and 150 CPU heads; existing main signal models/caches reused.',flush=True)
print('Earlier test outcomes were inspected. All comparisons remain exploratory.',flush=True)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()



def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()



def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)



def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)



def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)



def atomic_joblib(path, value):
    temporary = LOCAL / (Path(path).name + '.tmp')
    joblib.dump(value, temporary, compress=3)
    partial = Path(path).with_name(Path(path).name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)



def save_checkpoint(path, state):
    path = Path(path)
    temporary = LOCAL / (path.name + '.tmp')
    torch.save(state, temporary)
    partial = path.with_name(path.name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)



def load_checkpoint(path):
    return torch.load(path, map_location='cpu', weights_only=True)



def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)



def cuda_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()



class ActivityCNN(nn.Module):
    def __init__(self, channels, classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 7, padding=3), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.GroupNorm(8, 96), nn.GELU(),
            nn.Conv1d(96, 128, 5, stride=2, padding=2), nn.GroupNorm(8, 128), nn.GELU(),
            nn.AvgPool1d(kernel_size=8, stride=8), nn.Flatten(),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, classes))

    def forward(self, values, available=None):
        if available is None:
            available = torch.ones_like(values)
        return self.encoder(torch.cat([values, available], dim=1))



class SensorReconstructor(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 5, padding=2), nn.GELU(),
            nn.Conv1d(64, 96, 5, padding=2), nn.GELU(),
            nn.Conv1d(96, 64, 5, padding=2), nn.GELU(), nn.Conv1d(64, channels, 1))

    def forward(self, values, available):
        return self.network(torch.cat([values, available], dim=1))



def mask_groups(values, groups, rng):
    masked = np.array(values, dtype=np.float32, copy=True)
    available = np.ones_like(masked)
    for i in range(len(masked)):
        channels = groups[int(rng.integers(len(groups)))]['channels']
        masked[i, channels, :] = 0.0
        available[i, channels, :] = 0.0
    return masked, available



def classifier_epoch_inputs(clean, groups, seed):
    rng = np.random.default_rng(seed)
    values = np.array(clean, copy=True)
    available = np.ones_like(values)
    choices = rng.random(len(values))
    faults = np.flatnonzero((choices >= 0.35) & (choices < 0.80))
    masks = np.flatnonzero(choices >= 0.80)
    values[faults] = corrupt_signals(clean[faults], groups, rng)
    masked, indicators = mask_groups(clean[masks], groups, rng)
    values[masks], available[masks] = masked, indicators
    return values, available



def cpu_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}



@torch.inference_mode()
def action_outputs(classifier, reconstructor, values, groups, classes):
    classifier.eval()
    reconstructor.eval()
    number = len(values)
    group_count = len(groups)
    probabilities = np.empty((number, 1 + 2 * group_count, classes), dtype=np.float32)
    residuals = np.empty((number, group_count), dtype=np.float32)
    group_statistics = np.empty((number, group_count, 4), dtype=np.float32)
    cuda_sync()
    started = time.perf_counter()
    for start in range(0, number, INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, number)
        observed = torch.as_tensor(values[start:stop], device=DEVICE)
        probabilities[start:stop, 0] = classifier(observed).softmax(dim=1).cpu().numpy()
        for index, group in enumerate(groups):
            channels = group['channels']
            masked, available = observed.clone(), torch.ones_like(observed)
            masked[:, channels, :], available[:, channels, :] = 0.0, 0.0
            reconstruction = reconstructor(masked, available)
            repaired = observed.clone()
            repaired[:, channels, :] = reconstruction[:, channels, :]
            probabilities[start:stop, 1 + index] = classifier(masked, available).softmax(dim=1).cpu().numpy()
            probabilities[start:stop, 1 + group_count + index] = classifier(repaired).softmax(dim=1).cpu().numpy()
            residuals[start:stop, index] = (observed[:, channels, :] - reconstruction[:, channels, :]).square().mean(dim=(1, 2)).cpu().numpy()
            signal = values[start:stop, channels, :]
            group_statistics[start:stop, index] = np.stack([
                np.abs(signal).mean(axis=(1, 2)), signal.std(axis=(1, 2)),
                np.abs(np.diff(signal, axis=2)).mean(axis=(1, 2)),
                (signal == 0).mean(axis=(1, 2))], axis=1)
    cuda_sync()
    seconds = time.perf_counter() - started
    assert np.isfinite(probabilities).all() and np.isfinite(residuals).all()
    assert np.allclose(probabilities.sum(axis=2), 1.0, atol=1e-5)
    return probabilities, residuals, group_statistics, seconds



def corrupt_signals(values, groups, rng, fault=None, severity=None):
    # Fault locations and severity metadata are never returned as model features.
    result = np.array(values, dtype=np.float32, copy=True)
    for i in range(len(result)):
        kind = fault if fault is not None else str(rng.choice(['dropout', 'bias', 'gain']))
        level = int(severity) if severity is not None else int(rng.integers(0, 3))
        group = groups[int(rng.integers(len(groups)))]['channels']
        if kind == 'dropout':
            fraction = protocol['fault_severities']['dropout']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            result[i, group, start:start + length] = 0.0
        elif kind == 'bias':
            amount = protocol['fault_severities']['bias']['absolute_offset_in_training_std'][level]
            signs = rng.choice([-1.0, 1.0], size=len(group)).astype(np.float32)
            result[i, group, :] += np.float32(amount) * signs[:, None]
        elif kind == 'gain':
            factor = float(protocol['fault_severities']['gain']['factor'][level])
            factor = factor if rng.random() < 0.5 else 1.0 / factor
            result[i, group, :] *= np.float32(factor)
        elif kind == 'stuck':
            fraction = protocol['fault_severities']['stuck']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            first = result[i, group, start].copy()
            result[i, group, start:start + length] = first[:, None]
        else:
            raise ValueError(f'Unknown fault: {kind}')
    return result



def evaluate_choices(pack, choices, method):
    p, y = pack['probabilities'], pack['y']
    D, N, A, K = p.shape
    assert choices.shape == (D, N) and choices.min() >= 0 and choices.max() < A
    predictions = p.argmax(axis=-1)
    original_correct = predictions[:, :, 0] == y[None, :]
    selected = np.take_along_axis(predictions, choices[:, :, None], axis=2).squeeze(2)
    rows = []
    for i, condition in enumerate(pack['conditions']):
        correct = selected[i] == y
        helpful = float((~original_correct[i] & correct).mean())
        harmful = float((original_correct[i] & ~correct).mean())
        accuracy = float(correct.mean())
        assert abs(accuracy - original_correct[i].mean() - helpful + harmful) < 1e-10
        rows.append({'condition': condition, 'method': method, 'windows': N,
                     'macro_f1': macro_f1(y, selected[i], K), 'accuracy': accuracy,
                     'intervention_rate': float((choices[i] != 0).mean()), 'helpful_rate': helpful,
                     'harmful_rate': harmful,
                     'harm_given_original_correct': float(np.sum(original_correct[i] & ~correct) / max(1, original_correct[i].sum()))})
    return rows



def aggregate_rows(rows):
    clean = next(row for row in rows if row['condition'] == 'clean')
    corrupted = [row for row in rows if row['condition'] != 'clean']
    answer = {key: float(np.mean([row[key] for row in corrupted])) for key in
              ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}
    answer.update(clean_macro_f1=clean['macro_f1'], clean_accuracy=clean['accuracy'],
                  clean_harmful_rate=clean['harmful_rate'], clean_intervention_rate=clean['intervention_rate'])
    return answer



class PauseStudy(Exception):
    pass



def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)



def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names



def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())



def pack_from_caches(paths, expected_signature):
    matrices, residuals, statistics = [], [], []
    anchor, conditions = None, []
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == expected_signature
            identity = {key: saved[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key, value in identity.items():
                    assert np.array_equal(value, anchor[key]), f'Cache alignment error: {key}'
            matrices.append(saved['probabilities'])
            residuals.append(saved['reconstruction_residuals'])
            statistics.append(saved['observed_group_statistics'])
            conditions.append(str(saved['condition'].item()))
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and len(np.unique(pack['window_ids'])) == N
    return pack



def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'



def assert_table_parity(computed,original,keys,columns):
    assert not computed.duplicated(keys).any() and not original.duplicated(keys).any()
    a=computed.set_index(keys).sort_index()
    b=original.set_index(keys).sort_index()
    assert a.index.equals(b.index),'Table row identities changed.'
    for col in columns:
        assert np.allclose(a[col].to_numpy(dtype=float),b[col].to_numpy(dtype=float),atol=1e-12,rtol=0,equal_nan=True),f'Metric parity failed: {col}'



def verified_copy(source,relative,expected):
    assert not Path(relative).is_absolute() and '..' not in Path(relative).parts
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and file_sha256(target)==expected:return target
    partial=target.with_name(target.name+'.partial')
    shutil.copyfile(source,partial)
    assert file_sha256(partial)==expected,f'Input hash mismatch: {source}'
    os.replace(partial,target)
    return target



def stage_json(number,name):
    root=PROJECT/'outputs'/f'notebook{number:02d}_v1'
    inv={r['filename']:r for r in json.loads((root/f'notebook{number:02d}_artifact_inventory.json').read_text())}
    path=verified_copy(root/name,f'nb{number:02d}/{name}',inv[name]['sha256'])
    return json.loads(path.read_text()),inv[name]['sha256']



def load_source_data(job):
    dataset=job['dataset'];fold=job['fold']
    with np.load(prepared_file(dataset+'_signals.npz'),allow_pickle=False) as z:
        data={k:z[k] for k in ['X','y','subjects','window_ids']}
    with np.load(prepared_file(dataset+'_splits.npz'),allow_pickle=False) as z:
        splits={s:z[fold+'_'+s] for s in ['train','policy','validation','test']}
    all_rows=np.concatenate(list(splits.values()))
    assert np.array_equal(np.sort(all_rows),np.arange(len(data['y'])))
    for s,a in splits.items():
        for t,b in splits.items():
            if s!=t:assert not np.intersect1d(data['subjects'][a],data['subjects'][b]).size
    data['splits']=splits
    data['pool']=np.concatenate([splits['train'],splits['policy']])
    data['groups']=protocol['datasets'][dataset]['sensor_groups']
    data['classes']=len(protocol['datasets'][dataset]['class_names'])
    people=np.unique(data['subjects'][splits['train']])
    rng=np.random.default_rng(job['seed']+zlib.crc32((dataset+'|cohorts').encode()))
    people=rng.permutation(people)
    blocks=[b.tolist() for b in (np.array_split(people,len(people)) if dataset=='dsads' else np.array_split(people,5))]
    assert all(len(b)>0 for b in blocks)
    data['blocks']=blocks
    return data



def fit_auxiliary(kind,clean,y,groups,classes,run,signature,stage_seed):
    maximum=CLASSIFIER_EPOCHS if kind=='classifier' else RECONSTRUCTOR_EPOCHS
    seed_everything(stage_seed)
    model=(ActivityCNN(clean.shape[1],classes) if kind=='classifier' else SensorReconstructor(clean.shape[1])).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=.0001)
    path=run/(kind+'_last.pt');history=[];first=1
    if path.exists():
        cp=load_checkpoint(path)
        assert cp['signature']==signature and cp['maximum_epochs']==maximum
        model.load_state_dict(cp['model']);optimizer.load_state_dict(cp['optimizer'])
        history=cp['history'];first=cp['epoch']+1
        if first>maximum:
            print(f'  {kind}: reused fixed epoch {maximum}.',flush=True)
            return model
        torch.set_rng_state(cp['torch_rng'])
        if DEVICE.type=='cuda' and cp['cuda_rng']:torch.cuda.set_rng_state_all(cp['cuda_rng'])
        print(f'  {kind}: resuming epoch {first}.',flush=True)
    weights=1./np.sqrt(np.maximum(np.bincount(y,minlength=classes),1))
    weights=torch.tensor(weights/weights.mean(),dtype=torch.float32,device=DEVICE)
    for epoch in range(first,maximum+1):
        check_budget();started=time.perf_counter()
        lr=LEARNING_RATE*(.1+.9*.5*(1.+math.cos(math.pi*(epoch-1)/maximum)))
        for g in optimizer.param_groups:g['lr']=lr
        if kind=='classifier':
            values,available=classifier_epoch_inputs(clean,groups,stage_seed+epoch*1000)
            target=torch.from_numpy(y.astype(np.int64,copy=False))
        else:
            values,available=mask_groups(clean,groups,np.random.default_rng(stage_seed+epoch*1000))
            target=torch.from_numpy(clean)
        loader=DataLoader(TensorDataset(torch.from_numpy(values),torch.from_numpy(available),target),
            batch_size=BATCH_SIZE,shuffle=True,generator=torch.Generator().manual_seed(stage_seed+epoch),
            num_workers=0,pin_memory=DEVICE.type=='cuda')
        model.train();total=0.;seen=0
        for xb,mb,yb in loader:
            xb,mb,yb=xb.to(DEVICE),mb.to(DEVICE),yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True);out=model(xb,mb)
            if kind=='classifier':
                ew=weights[yb];loss=-(out.log_softmax(1).gather(1,yb[:,None]).squeeze(1)*ew).sum()/ew.sum()
            else:
                missing=1.-mb;loss=((out-yb).square()*missing).sum()/missing.sum().clamp_min(1.)
            assert torch.isfinite(loss),'Non-finite auxiliary loss.'
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step()
            total+=float(loss.item())*len(xb);seen+=len(xb)
        cuda_sync()
        history.append({'epoch':epoch,'loss':total/seen,'seconds':time.perf_counter()-started,'learning_rate':lr})
        save_checkpoint(path,{'signature':signature,'maximum_epochs':maximum,'epoch':epoch,'model':cpu_state(model),
            'optimizer':optimizer.state_dict(),'torch_rng':torch.get_rng_state(),
            'cuda_rng':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else [],'history':history})
        save_csv(run/(kind+'_history.csv'),pd.DataFrame(history))
        if epoch==1 or epoch%5==0 or epoch==maximum:
            print(f'  {kind} epoch {epoch:02d}/{maximum} | training loss {total/seen:.4f} | no validation selection',flush=True)
        del values,available,target,loader
    return model



def cache_valid(path,signature):
    digest=path.with_name(path.name+'.sha256.json')
    if not(path.exists() and digest.exists()):return False
    meta=json.loads(digest.read_text())
    assert meta['signature']==signature and meta['sha256']==file_sha256(path),'Saved auxiliary artifact changed.'
    return True



def subject_weights(subjects):
    people,inv,counts=np.unique(subjects,return_inverse=True,return_counts=True)
    return (1./counts[inv]).astype(np.float64)



def counterfactual_targets(predictions,y,subjects,classes):
    # Target is exact for ONE replaced prediction in a fixed keep confusion
    # matrix. It is a marginal utility surrogate, not additive/global F1.
    D,N,A=predictions.shape;J=A-1
    w=subject_weights(subjects);total=w.sum()
    weights=np.broadcast_to(w[None,:,None],(D,N,J)).copy()
    keep=predictions[:,:,0];actions=predictions[:,:,1:]
    correct=predictions==y[None,:,None]
    fix=(~correct[:,:,:1]&correct[:,:,1:]).astype(np.float32)
    harm=(correct[:,:,:1]&~correct[:,:,1:]).astype(np.float32)
    utility=np.zeros((D,N,J),dtype=np.float64)
    for d in range(D):
        C=np.bincount(y*classes+keep[d],weights=w,minlength=classes*classes).reshape(classes,classes)
        denominator=C.sum(0)+C.sum(1);tp=C.diagonal()
        old=np.divide(2.*tp,denominator,out=np.zeros(classes),where=denominator>0)
        b=np.broadcast_to(keep[d,:,None],(N,J));a=actions[d]
        wb=np.broadcast_to(w[:,None],(N,J))
        db=np.maximum(denominator[b]-wb,0.);da=denominator[a]+wb
        tpb=tp[b]-wb*(y[:,None]==b);tpa=tp[a]+wb*(y[:,None]==a)
        fb=np.divide(2.*tpb,db,out=np.zeros_like(db),where=db>1e-12)
        fa=np.divide(2.*tpa,da,out=np.zeros_like(da),where=da>1e-12)
        utility[d]=((fb-old[b])+(fa-old[a]))/classes*total/wb
        utility[d][a==b]=0.
    assert np.isfinite(utility).all()
    # Normalize sample weights to mean 1, preserving equal total mass per person.
    weights/=weights.mean()
    return utility.astype(np.float32),fix,harm,weights.astype(np.float32)



def fit_controller(name,X,target,weight,kind,run,signature,people):
    path=run/(name+'.joblib')
    if path.exists():
        payload=joblib.load(path)
        assert payload['signature']==signature and payload['training_participants']==people
        print('  controller',name,': reused.',flush=True)
        return payload['model']
    check_budget();start=time.perf_counter()
    if np.ptp(target)==0:
        model={'constant':float(target[0])}
    else:
        cls=HistGradientBoostingClassifier if kind=='binary' else HistGradientBoostingRegressor
        model=cls(learning_rate=.05,max_iter=MAX_ITER,max_leaf_nodes=15,max_depth=4,
            min_samples_leaf=40,l2_regularization=5.,max_bins=64,categorical_features=None,
            early_stopping=False,random_state=2026+zlib.crc32(name.encode())%10000)
        with threadpool_limits(limits=2):model.fit(X,target,sample_weight=weight)
    elapsed=time.perf_counter()-start
    atomic_joblib(path,{'signature':signature,'model':model,'kind':kind,'training_rows':len(target),
        'training_participants':people,'training_seconds':elapsed})
    print(f'  controller {name}: {len(target)} rows, {len(people)} people | {elapsed:.1f}s',flush=True)
    return model



def head_prediction(model,X,kind):
    if isinstance(model,dict):return np.full(len(X),model['constant'],dtype=np.float32)
    with threadpool_limits(limits=2):
        if kind=='binary':return model.predict_proba(X)[:,list(model.classes_).index(1)].astype(np.float32)
        return model.predict(X).astype(np.float32)



def scored_actions(scores,threshold):
    if threshold is None:return np.zeros(scores.shape[:2],dtype=np.int64)
    return np.where(scores.max(-1)>threshold,scores.argmax(-1)+1,0)



def select_family(pack,scores_by_penalty,family):
    base=aggregate_rows(evaluate_choices(pack,np.zeros(scores_by_penalty[1.].shape[:2],dtype=np.int64),'keep'))
    best=None;trials=[]
    for penalty in HARM_PENALTIES:
        for threshold in THRESHOLDS:
            actions=scored_actions(scores_by_penalty[penalty],threshold)
            summary=aggregate_rows(evaluate_choices(pack,actions,family))
            feasible=(summary['clean_macro_f1']>=base['clean_macro_f1']-CLEAN_F1_TOLERANCE-1e-12
                and summary['clean_harmful_rate']<=MAX_HARM+1e-12 and summary['harmful_rate']<=MAX_HARM+1e-12)
            trials.append({'family':family,'harm_penalty':penalty,'threshold':threshold,'feasible':feasible,**summary})
            rank=(summary['macro_f1'],summary['clean_macro_f1'],-summary['harmful_rate'],-summary['intervention_rate'],threshold is None)
            if feasible and (best is None or rank>best['rank']):
                best={'rank':rank,'settings':{'family':family,'harm_penalty':penalty,'threshold':threshold},
                      'summary':summary,'actions':actions,'structural_keep_only':threshold is None}
    assert best is not None
    del best['rank']
    return best,trials



def auxiliary_cohort(job,data,number,people,run):
    folder=run/'auxiliary'/f'cohort_{number:02d}';folder.mkdir(parents=True,exist_ok=True)
    held=data['pool'][np.isin(data['subjects'][data['pool']],people)]
    train=data['pool'][~np.isin(data['subjects'][data['pool']],people)]
    tr_people=np.unique(data['subjects'][train]).tolist()
    assert not set(tr_people)&set(people)
    assert not np.intersect1d(np.concatenate([train,held]),data['splits']['validation']).size
    assert not np.intersect1d(np.concatenate([train,held]),data['splits']['test']).size
    audit={'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'cohort':number,
       'training_participants':tr_people,'held_participants':people,'training_windows':len(train),
       'held_windows':len(held),'train_held_participant_overlap':0,'normalization_fit':'auxiliary training rows only',
       'checkpoint_selection':'fixed final epoch; no validation or held-source scoring',
       'source_indices_sha256':hashlib.sha256(train.tobytes()+held.tobytes()).hexdigest()}
    signature=json_signature({'study':STUDY_SIGNATURE,**audit})
    audit['signature']=signature
    split_path=folder/'source_split.json'
    if split_path.exists():assert json.loads(split_path.read_text())==audit
    else:save_json(split_path,audit)
    paths=[folder/(c+'.npz') for c in DEV_CONDITIONS]
    if all(cache_valid(p,signature) for p in paths):
        print(f'  auxiliary cohort {number}: reused all 10 conditions.',flush=True)
        return pack_from_caches(paths,signature),audit
    if REQUIRE_GPU and DEVICE.type!='cuda':
        raise RuntimeError('Auxiliary training/caches remain unfinished. Select Runtime > Change runtime type > T4 GPU, then rerun the same cell.')
    check_budget()
    raw=data['X'][train]
    mean=raw.mean(axis=(0,2),dtype=np.float64).astype(np.float32)
    std=np.maximum(raw.std(axis=(0,2),dtype=np.float64),1e-6).astype(np.float32)
    save_npz(folder/'normalization.npz',mean=mean,std=std,fit_source_indices=train,signature=np.asarray(signature))
    clean=np.ascontiguousarray((raw-mean[None,:,None])/std[None,:,None],dtype=np.float32)
    observed=np.ascontiguousarray((data['X'][held]-mean[None,:,None])/std[None,:,None],dtype=np.float32)
    groups=data['groups'];K=data['classes']
    stage_seed=job['seed']+zlib.crc32((job['dataset']+'|aux|'+str(number)).encode())%1000000
    print(f'  Auxiliary cohort {number}: train people {tr_people}; predict held people {people}.',flush=True)
    classifier=fit_auxiliary('classifier',clean,data['y'][train],groups,K,folder,signature,stage_seed)
    reconstructor=fit_auxiliary('reconstructor',clean,data['y'][train],groups,K,folder,signature,stage_seed+100)
    names=np.asarray(['keep']+[f'mask:{g["name"]}' for g in groups]+[f'reconstruct:{g["name"]}' for g in groups])
    for condition,path in zip(DEV_CONDITIONS,paths):
        if cache_valid(path,signature):continue
        check_budget()
        values=observed
        if condition!='clean':
            fault,level=condition.rsplit('_',1)
            rng=np.random.default_rng(stage_seed+zlib.crc32(condition.encode()))
            values=corrupt_signals(observed,groups,rng,fault,int(level)-1)
        p,r,s,seconds=action_outputs(classifier,reconstructor,values,groups,K)
        save_npz(path,signature=np.asarray(signature),condition=np.asarray(condition),split=np.asarray('oof'),
          y=data['y'][held],subjects=data['subjects'][held],window_ids=data['window_ids'][held],source_indices=held,
          action_names=names,probabilities=p,reconstruction_residuals=r,observed_group_statistics=s)
        save_json(path.with_name(path.name+'.sha256.json'),{'signature':signature,'sha256':file_sha256(path)})
        print(f'  held source {condition}: {len(held)} windows | {seconds:.1f}s',flush=True)
        del p,r,s,values
    del classifier,reconstructor,clean,observed,raw
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return pack_from_caches(paths,signature),audit



def combine_packs(packs):
    first=packs[0]
    for p in packs:
        assert p['conditions']==DEV_CONDITIONS and np.array_equal(p['action_names'],first['action_names'])
    out={k:np.concatenate([p[k] for p in packs]) for k in ['y','subjects','window_ids','source_indices']}
    out.update({k:np.concatenate([p[k] for p in packs],axis=1) for k in ['probabilities','residuals','statistics']})
    out.update(action_names=first['action_names'],conditions=DEV_CONDITIONS)
    assert len(np.unique(out['source_indices']))==len(out['y'])
    assert len(np.unique(out['window_ids']))==len(out['y'])
    return out



def feature_matrix(pack,configuration):
    scale=np.asarray(configuration['residual_scale'],dtype=np.float32)
    X,names=policy_features(pack['probabilities'],pack['residuals'],pack['statistics'],scale)
    assert names==configuration['feature_names']
    return X



def original_scores(job,pack,configuration):
    X=feature_matrix(pack,configuration)
    D,N,A,K=pack['probabilities'].shape;shape=(D,N,A-1)
    def pred(name,kind):return head_prediction(original_head(job,name,configuration),X,kind).reshape(shape)
    pf=pred('pooled_fix','binary');ph=pred('pooled_harm','binary');pu=pred('pooled_f1','regression')
    su=np.stack([pred(f'jackknife_{b}_f1','regression') for b in range(3)])
    sh=np.stack([pred(f'jackknife_{b}_harm','binary') for b in range(3)])
    scores={f:{} for f in ORIGINAL_FAMILIES}
    for penalty in HARM_PENALTIES:
        scores['oof_accuracy_utility'][penalty]=pf-penalty*ph
        scores['oof_f1_utility'][penalty]=pu-(penalty-1.)*ph
        s=su-(penalty-1.)*sh
        scores['source_mean_f1'][penalty]=s.mean(0)
        scores['source_min_f1'][penalty]=s.min(0)
    return scores



def stratified_window_subset(subjects,count,seed):
    people,counts=np.unique(subjects,return_counts=True)
    assert len(people)<=count<=len(subjects)
    expected=counts.astype(float)*count/len(subjects)
    allocations=np.minimum(counts,np.maximum(np.floor(expected).astype(int),1))
    while allocations.sum()<count:
        eligible=np.flatnonzero(allocations<counts)
        pick=eligible[np.argmax((expected-allocations)[eligible])];allocations[pick]+=1
    while allocations.sum()>count:
        eligible=np.flatnonzero(allocations>1)
        pick=eligible[np.argmax((allocations-expected)[eligible])];allocations[pick]-=1
    rng=np.random.default_rng(seed);mask=np.zeros(len(subjects),dtype=bool)
    for person,n in zip(people,allocations):
        candidates=np.flatnonzero(subjects==person)
        mask[rng.choice(candidates,size=int(n),replace=False)]=True
    assert int(mask.sum())==count and np.array_equal(np.unique(subjects[mask]),people)
    return mask



def fit_matched_head(block,kind,X,target,weight,run,audit):
    suffix='f1' if kind=='regression' else 'harm'
    name=f'window_{block}_{suffix}';path=run/(name+'.joblib')
    seed=int(2026+zlib.crc32(f'jackknife_{block}_{suffix}'.encode())%10000)
    if path.exists():
        saved=joblib.load(path)
        assert saved['signature']==audit['signature'] and saved['random_state']==seed
        assert saved['training_rows']==audit['action_rows']
        print('  matched head',name,': reused.',flush=True)
        return saved['model']
    check_budget();start=time.perf_counter()
    if np.ptp(target)==0:model={'constant':float(target[0])}
    else:
        cls=HistGradientBoostingRegressor if kind=='regression' else HistGradientBoostingClassifier
        model=cls(learning_rate=.05,max_iter=MAX_ITER,max_leaf_nodes=15,max_depth=4,
          min_samples_leaf=40,l2_regularization=5.,max_bins=64,categorical_features=None,
          early_stopping=False,random_state=seed)
        with threadpool_limits(limits=2):model.fit(X,target,sample_weight=weight)
    seconds=time.perf_counter()-start
    atomic_joblib(path,{'signature':audit['signature'],'model':model,'kind':kind,'random_state':seed,
      'training_rows':len(target),'training_participants':audit['selected_participants'],'training_seconds':seconds})
    print(f'  matched head {name}: {len(target)} rows | {seconds:.1f}s',flush=True)
    return model



def control_scores(pack,configuration,run,audits):
    X=feature_matrix(pack,configuration);D,N,A,K=pack['probabilities'].shape
    utility=[];harm=[]
    for b in range(3):
        audit=next(a for a in audits if a['block']==b)
        def pred(suffix,kind):
            payload=joblib.load(run/f'window_{b}_{suffix}.joblib')
            assert payload['signature']==audit['signature']
            return head_prediction(payload['model'],X,kind).reshape(D,N,A-1)
        utility.append(pred('f1','regression'));harm.append(pred('harm','binary'))
    utility=np.stack(utility);harm=np.stack(harm)
    result={f:{} for f in CONTROL_FAMILIES}
    for penalty in HARM_PENALTIES:
        s=utility-(penalty-1.)*harm
        result['window_mean_f1'][penalty]=s.mean(0)
        result['window_min_f1'][penalty]=s.min(0)
    return result



def assert_saved_artifacts(run,result):
    assert result['study_signature']==STUDY_SIGNATURE
    for item in result['artifacts']:assert file_sha256(run/item['filename'])==item['sha256'],item['filename']



def local_artifacts(run,names):
    return [{'filename':name,'sha256':file_sha256(run/name)} for name in names]



def condition_rows(job,y,predictions,conditions,classes,actions=None):
    base=predictions[0];rows=[]
    assert predictions.shape==(len(METHODS),len(conditions),len(y))
    for m,method in enumerate(METHODS):
        for d,condition in enumerate(conditions):
            row={**job,'method':method,'condition':condition,'windows':len(y),**metrics(y,predictions[m,d],base[d],classes)}
            row['sensor_action_intervention_rate']=(float((actions[NEW_METHODS.index(method),d]!=0).mean())
                 if actions is not None and method in NEW_METHODS else (0. if method=='keep' else np.nan))
            rows.append(row)
    return pd.DataFrame(rows)



def participant_rows(job,y,subjects,predictions,classes):
    rows=[]
    for m,method in enumerate(METHODS):
        if method=='oracle_diagnostic':continue
        for person in np.unique(subjects):
            use=subjects==person;present=np.unique(y[use])
            for scope,conditions in SCOPES.items():
                values=[metrics(y[use],predictions[m,CONDITIONS.index(c),use],predictions[0,CONDITIONS.index(c),use],classes,present) for c in conditions]
                average={k:float(np.mean([v[k] for v in values])) for k in METRICS}
                average['macro_f1_present_classes']=average.pop('macro_f1')
                rows.append({**job,'method':method,'subject':int(person),'scope':scope,'windows':int(use.sum()),
                    'classes_present':len(present),**average})
    return pd.DataFrame(rows)



def phase_b(job,selection):
    run=OUTPUT/'runs'/identity(job);done=run/'test_complete.json'
    if done.exists():
        r=json.loads(done.read_text());assert_saved_artifacts(run,r)
        print('  Reused test evaluation:',identity(job),flush=True)
        return r
    check_budget();print('\nPHASE B',identity(job),flush=True)
    assert_saved_artifacts(run,selection)
    frozen=json.loads((run/'frozen_policies.json').read_text())
    pack=load_pack(job,'test');configuration=frozen['configuration']
    assert not np.intersect1d(pack['subjects'],configuration['source_participants']+configuration['validation_participants']).size
    scores=original_scores(job,pack,configuration)
    scores.update(control_scores(pack,configuration,run,frozen['matched_control_audit']))
    actions=[]
    for family in NEW_METHODS:
        item=frozen['selected_methods'][family];settings=item['settings']
        a=scored_actions(scores[family][settings['harm_penalty']],settings['threshold'])
        if item['structural_keep_only']:assert not a.any()
        actions.append(a)
    actions=np.stack(actions)
    candidates=pack['probabilities'].argmax(-1)
    new_pred=np.stack([np.take_along_axis(candidates,a[:,:,None],axis=2).squeeze(2) for a in actions])
    with np.load(source_run_file(9,job,'test_predictions.npz'),allow_pickle=False) as z:
        assert z['signature'].item()==RESULTS9[identity(job)]['signature']
        assert z['methods'].tolist()==OLD_METHODS and z['conditions'].tolist()==CONDITIONS
        for key in ['y','subjects','window_ids','source_indices']:assert np.array_equal(z[key],pack[key])
        old_pred=z['predictions']
    assert np.array_equal(old_pred[0],candidates[:,:,0]),'Keep predictions differ between saved baseline and candidate caches.'
    predictions=np.concatenate([old_pred,new_pred])
    K=pack['probabilities'].shape[-1]
    assert predictions.min()>=0 and predictions.max()<K
    rows=condition_rows(job,pack['y'],predictions,CONDITIONS,K,actions)
    original=pd.read_csv(source_run_file(9,job,'per_condition.csv'))
    assert_table_parity(rows[rows.method.isin(OLD_METHODS)],original,
       ['dataset','fold','seed','method','condition'],METRICS)
    participants=participant_rows(job,pack['y'],pack['subjects'],predictions,K)
    summaries=[]
    for scope,conditions in SCOPES.items():
        frame=rows[rows.condition.isin(conditions)].groupby(['dataset','fold','seed','method'],sort=False)[METRICS+['sensor_action_intervention_rate']].mean().reset_index()
        frame['scope']=scope;summaries.append(frame)
    summary=pd.concat(summaries,ignore_index=True)
    save_csv(run/'per_condition.csv',rows);save_csv(run/'per_participant.csv',participants);save_csv(run/'test_summary.csv',summary)
    save_npz(run/'test_predictions.npz',study_signature=np.asarray(STUDY_SIGNATURE),methods=np.asarray(METHODS),
      conditions=np.asarray(CONDITIONS),predictions=predictions,new_action_methods=np.asarray(NEW_METHODS),
      new_actions=actions,y=pack['y'],subjects=pack['subjects'],window_ids=pack['window_ids'],source_indices=pack['source_indices'])
    result={**job,'study_signature':STUDY_SIGNATURE,'test_windows':len(pack['y']),
      'test_participants':np.unique(pack['subjects']).tolist(),'original_test_metric_parity_passed':True,
      'original_threshold_changes':0,'matched_control_thresholds_selected_on':'validation only',
      'artifacts':local_artifacts(run,['per_condition.csv','per_participant.csv','test_summary.csv','test_predictions.npz'])}
    save_json(done,result)
    print(f'  Reproduced all 22 previous methods; evaluated all six action controllers on {len(pack["y"])} test windows.',flush=True)
    del scores,pack,predictions,new_pred
    gc.collect()
    return result



def build_audits(summary,conditions,participants):
    comparisons=[];harm=[]
    for (dataset,fold,seed,scope),group in summary.groupby(['dataset','fold','seed','scope'],sort=False):
        lookup={r['method']:r for r in group.to_dict('records')}
        for focal in [PRIMARY,'source_min_f1','oof_f1_utility']:
            for comparator,row in lookup.items():
                if comparator in [focal,'oracle_diagnostic']:continue
                comparisons.append({'dataset':dataset,'fold':fold,'seed':seed,'scope':scope,'method':focal,
                  'comparator':comparator,'primary_method':focal==PRIMARY,
                  'policy_macro_f1_pct':lookup[focal]['macro_f1']*100,'comparator_macro_f1_pct':row['macro_f1']*100,
                  'delta_macro_f1_pp':(lookup[focal]['macro_f1']-row['macro_f1'])*100,
                  'policy_harmful_rate_pct':lookup[focal]['harmful_rate']*100,
                  'comparator_harmful_rate_pct':row['harmful_rate']*100})
        for method,row in lookup.items():
            part=conditions[(conditions.dataset==dataset)&(conditions.fold==fold)&(conditions.seed==seed)&(conditions.method==method)&conditions.condition.isin(SCOPES[scope])]
            people=participants[(participants.dataset==dataset)&(participants.fold==fold)&(participants.seed==seed)&(participants.method==method)&(participants.scope==scope)]
            record={'dataset':dataset,'fold':fold,'seed':seed,'scope':scope,'method':method,
              'macro_f1_pct':row['macro_f1']*100,'mean_harm_pct':row['harmful_rate']*100,
              'mean_harm_le_1pct':bool(row['harmful_rate']<=.01+1e-12),
              'maximum_condition_harm_pct':float(part.harmful_rate.max()*100),
              'conditions_exceeding_1pct_harm':int((part.harmful_rate>.01+1e-12).sum()),
              'worst_condition_macro_f1_pct':float(part.macro_f1.min()*100),
              'maximum_participant_mean_harm_pct':float(people.harmful_rate.max()*100) if len(people) else np.nan,
              'participants_exceeding_1pct_mean_harm':int((people.harmful_rate>.01+1e-12).sum()) if len(people) else np.nan,
              'test_participants':len(people),'descriptive_only':True}
            if scope=='clean':
                loss=(lookup['keep']['macro_f1']-row['macro_f1'])*100
                record.update(clean_f1_loss_vs_keep_pp=loss,clean_f1_loss_le_0_5pp=bool(loss<=.5+1e-12))
            harm.append(record)
    return pd.DataFrame(comparisons),pd.DataFrame(harm)



def control_mask(job,pack,block,run):
    D,N,A,K=pack['probabilities'].shape;J=A-1
    matched=next(a for a in json.loads((run/'jackknife_audit.json').read_text()) if a['block']==block)
    rows=matched['training_rows'];assert rows%(D*J)==0
    count=int(rows//(D*J))
    seed=2026+zlib.crc32((identity(job)+'|window-control|'+str(block)).encode())
    mask=stratified_window_subset(pack['subjects'],count,seed)
    audit={'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'block':block,
      'source_participants':np.unique(pack['subjects']).tolist(),
      'selected_participants':np.unique(pack['subjects'][mask]).tolist(),
      'source_windows':N,'selected_windows':count,'action_rows':int(count*D*J),
      'matched_source_jackknife_action_rows':rows,'head_iterations':MAX_ITER,
      'subset_seed':int(seed),'selection_unit':'original window; all condition/action views retained',
      'window_ids_sha256':json_signature(pack['window_ids'][mask].tolist()),
      'selected_counts_by_participant':{str(int(s)):int(np.sum(pack['subjects'][mask]==s)) for s in np.unique(pack['subjects'])}}
    signature=json_signature({'study':STUDY_SIGNATURE,**audit});audit['signature']=signature
    path=run/f'window_{block}_subset.npz'
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            assert z['signature'].item()==signature and np.array_equal(z['mask'],mask)
    else:save_npz(path,signature=np.asarray(signature),mask=mask,source_indices=pack['source_indices'][mask],window_ids=pack['window_ids'][mask])
    return mask,audit



def metrics(y,pred,base,K,present=None):
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    den=cm.sum(0)+cm.sum(1)
    f1=np.divide(2.*cm.diagonal(),den,out=np.zeros(K),where=den>0)
    correct=pred==y;was_correct=base==y
    helpful=float((~was_correct & correct).mean());harmful=float((was_correct & ~correct).mean())
    assert abs(correct.mean()-was_correct.mean()-helpful+harmful)<1e-12
    return {'macro_f1':float(f1.mean() if present is None else f1[present].mean()),'accuracy':float(correct.mean()),
            'decision_change_rate':float((pred!=base).mean()),'helpful_rate':helpful,'harmful_rate':harmful}



def check_budget():
    if time.monotonic()>=SESSION_DEADLINE:
        raise PauseStudy('Session budget reached; finished epochs, caches, head fits and phases are saved.')



def prepared_file(name):
    return verified_copy(PREPARED/name,'prepared/'+name,prep_inventory[name]['sha256'])



def load_pack(job,split):
    assert split in ['policy','validation','test']
    if split=='test':assert (OUTPUT/'notebook14_frozen_test_plan.json').exists(),'Freeze all ten new validation selections first.'
    conditions=CONDITIONS if split=='test' else DEV_CONDITIONS
    paths=[];signature=None
    for condition in conditions:
        item=CACHE_INPUTS[identity(job)][split+'|'+condition]
        if signature is None:signature=item['signature']
        assert signature==item['signature']
        path=verified_copy(STAGES[item['stage']]/item['filename'],item['stage']+'/'+item['filename'],item['sha256'])
        with np.load(path,allow_pickle=False) as z:
            assert z['condition'].item()==condition and z['split'].item()==split
        paths.append(path)
    pack=pack_from_caches(paths,signature);assert pack['conditions']==conditions
    return pack



def source_ready(job):
    run=OUTPUT/'runs'/identity(job);done=run/'source_ready.json'
    if not done.exists():return None
    result=json.loads(done.read_text());assert_saved_artifacts(run,result)
    return result



def generate_sources(job):
    run=OUTPUT/'runs'/identity(job);run.mkdir(parents=True,exist_ok=True)
    previous=source_ready(job)
    if previous is not None:
        print('  Source caches already complete:',identity(job),flush=True)
        return previous
    assert str(torch.__version__)==recipe['software']['torch'],(
        f'Auxiliary neural fitting requires PyTorch {recipe["software"]["torch"]}, matching Notebook 11; '
        f'this runtime has {torch.__version__}. CPU controller fitting may use a different PyTorch build.')
    check_budget();print('\nGPU SOURCE STAGE',identity(job),flush=True)
    data=load_source_data(job)
    assert len(data['blocks'])==(4 if job['dataset']=='dsads' else 5)
    assert len(np.unique(data['subjects'][data['pool']]))==(5 if job['dataset']=='dsads' else 18)
    original=load_pack(job,'policy')
    assert np.array_equal(original['source_indices'],data['splits']['policy'])
    for key in ['y','subjects','window_ids']:
        assert np.array_equal(original[key],data[key][original['source_indices']])
    audits=[{'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'cohort':'original_policy',
      'training_participants':np.unique(data['subjects'][data['splits']['train']]).tolist(),
      'held_participants':np.unique(original['subjects']).tolist(),'training_windows':len(data['splits']['train']),
      'held_windows':len(original['y']),'train_held_participant_overlap':0,
      'normalization_fit':'original train only','checkpoint_selection':'original frozen validation-selected checkpoint'}]
    files=[]
    for i,people in enumerate(data['blocks']):
        pack,audit=auxiliary_cohort(job,data,i,people,run);audits.append(audit)
        assert set(np.unique(pack['subjects']).tolist())==set(people)
        folder=f'auxiliary/cohort_{i:02d}/'
        files.extend(folder+c+'.npz' for c in DEV_CONDITIONS)
        files.extend(folder+c for c in ['source_split.json','normalization.npz','classifier_history.csv','reconstructor_history.csv'])
        del pack
    save_json(run/'source_audit.json',audits)
    result={**job,'study_signature':STUDY_SIGNATURE,'auxiliary_torch_version':str(torch.__version__),
      'source_participants':np.unique(data['subjects'][data['pool']]).tolist(),
      'validation_participants':np.unique(data['subjects'][data['splits']['validation']]).tolist(),
      'test_participants':np.unique(data['subjects'][data['splits']['test']]).tolist(),
      'source_windows':len(data['pool']),'source_indices_sha256':hashlib.sha256(np.sort(data['pool']).astype(np.int64).tobytes()).hexdigest(),
      'classes':data['classes'],'source_audit':audits,'artifacts':local_artifacts(run,files+['source_audit.json'])}
    save_json(run/'source_ready.json',result)
    del data,original
    gc.collect()
    return result



def pooled_source_pack(job,ready):
    run=OUTPUT/'runs'/identity(job);packs=[load_pack(job,'policy')]
    for a in ready['source_audit']:
        if a['cohort']=='original_policy':continue
        paths=[run/'auxiliary'/f'cohort_{a["cohort"]:02d}'/(c+'.npz') for c in DEV_CONDITIONS]
        pack=pack_from_caches(paths,a['signature'])
        assert np.unique(pack['subjects']).tolist()==sorted(a['held_participants'])
        packs.append(pack)
    pooled=combine_packs(packs)
    assert len(pooled['y'])==ready['source_windows']
    assert np.unique(pooled['subjects']).tolist()==ready['source_participants']
    assert hashlib.sha256(np.sort(pooled['source_indices']).astype(np.int64).tobytes()).hexdigest()==ready['source_indices_sha256']
    return pooled



def original_head(job,name,configuration):
    payload=joblib.load(OUTPUT/'runs'/identity(job)/(name+'.joblib'))
    assert payload['signature']==configuration['signature']
    model=payload['model']
    if not isinstance(model,dict):assert model.get_params()['max_iter']==MAX_ITER
    return model



def fit_policies(job,ready):
    run=OUTPUT/'runs'/identity(job);done=run/'validation_complete.json'
    if done.exists():
        result=json.loads(done.read_text());assert_saved_artifacts(run,result)
        print('  Reused frozen validation:',identity(job),flush=True)
        return result
    check_budget();print('\nCPU CONTROLLER STAGE',identity(job),flush=True)
    source=pooled_source_pack(job,ready);validation=load_pack(job,'validation')
    assert np.unique(validation['subjects']).tolist()==ready['validation_participants']
    assert not np.intersect1d(source['subjects'],validation['subjects']).size
    assert not np.intersect1d(source['subjects'],ready['test_participants']).size
    assert not np.intersect1d(source['window_ids'],validation['window_ids']).size
    original_policy=load_pack(job,'policy')
    scale=np.maximum(np.quantile(original_policy['residuals'][0],.95,axis=0),1e-8).astype(np.float32)
    X,names=policy_features(source['probabilities'],source['residuals'],source['statistics'],scale)
    signature=json_signature({'study':STUDY_SIGNATURE,'job':job,'source_people':ready['source_participants'],'features':names})
    configuration={'signature':signature,'feature_names':names,'residual_scale':scale.tolist(),
       'source_participants':ready['source_participants'],'validation_participants':ready['validation_participants'],
       'test_participants':ready['test_participants'],'source_windows':len(source['y'])}
    save_json(run/'controller_inputs.json',configuration)
    pred=source['probabilities'].argmax(-1);D,N,A,K=source['probabilities'].shape;J=A-1
    utility,fix,harm,w=counterfactual_targets(pred,source['y'],source['subjects'],K)
    for name,target,kind in [('pooled_fix',fix,'binary'),('pooled_harm',harm,'binary'),('pooled_f1',utility,'regression')]:
        fit_controller(name,X,target.reshape(-1),w.reshape(-1),kind,run,signature,ready['source_participants'])
    del utility,fix,harm,w
    blocks=np.array_split(np.random.default_rng(2026).permutation(ready['source_participants']),3)
    jackknife=[]
    for b,excluded in enumerate(blocks):
        mask=~np.isin(source['subjects'],excluded)
        t,_,h,w=counterfactual_targets(pred[:,mask],source['y'][mask],source['subjects'][mask],K)
        rows=np.broadcast_to(mask[None,:,None],(D,N,J)).reshape(-1);subX=X[rows]
        people=np.unique(source['subjects'][mask]).tolist()
        fit_controller(f'jackknife_{b}_f1',subX,t.reshape(-1),w.reshape(-1),'regression',run,signature,people)
        fit_controller(f'jackknife_{b}_harm',subX,h.reshape(-1),w.reshape(-1),'binary',run,signature,people)
        jackknife.append({'block':b,'excluded_people':excluded.tolist(),'training_people':people,
          'targets_recomputed_without_excluded_people':True,'training_rows':int(mask.sum()*D*J)})
        del subX,t,h,w
    save_json(run/'jackknife_audit.json',jackknife)
    audits=[]
    for b in range(3):
        mask,audit=control_mask(job,source,b,run);audits.append(audit)
        t,_,h,w=counterfactual_targets(pred[:,mask],source['y'][mask],source['subjects'][mask],K)
        rows=np.broadcast_to(mask[None,:,None],(D,N,J)).reshape(-1);subX=X[rows]
        fit_matched_head(b,'regression',subX,t.reshape(-1),w.reshape(-1),run,audit)
        fit_matched_head(b,'binary',subX,h.reshape(-1),w.reshape(-1),run,audit)
        del subX,t,h,w
    save_json(run/'matched_control_audit.json',audits)
    scores=original_scores(job,validation,configuration)
    scores.update(control_scores(validation,configuration,run,audits))
    selected={};decisions=[];trials=[];rows=[]
    for family in NEW_METHODS:
        item,grid=select_family(validation,scores[family],family)
        decisions.append(item.pop('actions'));selected[family]=item;trials.extend(grid)
        rows.append({**job,'method':family,'structural_keep_only':item['structural_keep_only'],**item['summary']})
        print('  Selected',family,':',item['settings'],flush=True)
    baseline_actions={'keep':np.zeros(validation['probabilities'].shape[:2],dtype=np.int64),
       'confidence_select':validation['probabilities'].max(-1).argmax(-1)}
    for name,actions in baseline_actions.items():
        rows.append({**job,'method':name,'structural_keep_only':name=='keep',**aggregate_rows(evaluate_choices(validation,actions,name))})
    save_csv(run/'validation_summary.csv',pd.DataFrame(rows));save_csv(run/'validation_trials.csv',pd.DataFrame(trials))
    save_npz(run/'validation_decisions.npz',methods=np.asarray(NEW_METHODS),conditions=np.asarray(DEV_CONDITIONS),
      actions=np.stack(decisions),y=validation['y'],subjects=validation['subjects'],window_ids=validation['window_ids'],source_indices=validation['source_indices'])
    frozen={'study_signature':STUDY_SIGNATURE,'job':job,'configuration':configuration,'primary_method':PRIMARY,
      'selected_methods':selected,'matched_control_audit':audits}
    save_json(run/'frozen_policies.json',frozen)
    files=['controller_inputs.json','jackknife_audit.json','matched_control_audit.json','validation_summary.csv',
      'validation_trials.csv','validation_decisions.npz','frozen_policies.json']
    files += ['pooled_fix.joblib','pooled_harm.joblib','pooled_f1.joblib']
    files += [f'jackknife_{b}_{kind}.joblib' for b in range(3) for kind in ['f1','harm']]
    files += [f'window_{b}_{kind}.joblib' for b in range(3) for kind in ['f1','harm']]
    files += [f'window_{b}_subset.npz' for b in range(3)]
    result={**job,'study_signature':STUDY_SIGNATURE,'source_participants':ready['source_participants'],
      'validation_participants':ready['validation_participants'],'validation_windows':len(validation['y']),
      'selected_methods':selected,'matched_control_audit':audits,'jackknife_audit':jackknife,
      'artifacts':local_artifacts(run,files)}
    save_json(done,result)
    del source,validation,original_policy,X,pred,scores
    gc.collect()
    return result



PREPARED=PROJECT/'processed/v1'
SOURCE9=PROJECT/'outputs/notebook09_v1'
SOURCE11=PROJECT/'outputs/notebook11_v1'
SOURCE12=PROJECT/'outputs/notebook12_v1'
SOURCE13=PROJECT/'outputs/notebook13_v1'
STAGES={f'nb{i:02d}':PROJECT/'outputs'/f'notebook{i:02d}_v1' for i in [2,4,5]}
reports={};manifests={};report_hashes={};manifest_hashes={}
for number in [9,11,12,13]:
    report,rhash=stage_json(number,f'notebook{number:02d}_run_report.json')
    manifest,mhash=stage_json(number,f'notebook{number:02d}_study_manifest.json')
    assert report['study_complete'],f'Finish Notebook {number:02d} first.'
    assert report['study_signature']==manifest['signature']==json_signature(manifest['plan'])
    reports[number]=report;manifests[number]=manifest
    report_hashes[f'nb{number:02d}']=rhash;manifest_hashes[f'nb{number:02d}']=mhash
report9,report11,report12,report13=[reports[i] for i in [9,11,12,13]]
manifest9,manifest11,manifest12,manifest13=[manifests[i] for i in [9,11,12,13]]
assert manifest11['plan']['source_manifest_sha256']==manifest_hashes['nb09']
assert manifest12['plan']['source11_report_sha256']==report_hashes['nb11']
assert manifest12['plan']['source9_report_sha256']==report_hashes['nb09']
for key in ['nb09','nb11','nb12']:
    assert manifest13['plan']['source_report_hashes'][key]==report_hashes[key]
assert report12['primary_method']==report13['primary_method']==PRIMARY
assert report13['seed_count_per_dataset']==1 and report13['methods_reported']==len(METHODS)
recipe=manifest11['plan']
assert sklearn.__version__==recipe['software']['sklearn'],(
    f'Use scikit-learn {recipe["software"]["sklearn"]}, as in Notebook 11; current version is {sklearn.__version__}.')
CLASSIFIER_EPOCHS=recipe['auxiliary_training']['classifier_epochs']
RECONSTRUCTOR_EPOCHS=recipe['auxiliary_training']['reconstructor_epochs']
assert BATCH_SIZE==recipe['auxiliary_training']['batch_size']
assert LEARNING_RATE==recipe['auxiliary_training']['learning_rate']
MAX_ITER=recipe['controller_head_iterations']
HARM_PENALTIES=recipe['validation_grid']['harm_penalty'];THRESHOLDS=recipe['validation_grid']['threshold']
CLEAN_F1_TOLERANCE=recipe['validation_guards']['clean_f1_loss'];MAX_HARM=recipe['validation_guards']['clean_and_mean_corrupt_harm']
assert recipe['controller_families']==ORIGINAL_FAMILIES
assert HARM_PENALTIES==[1.,2.,4.] and THRESHOLDS[-1] is None
recipe_subset={'auxiliary_training':recipe['auxiliary_training'],'controller_head_iterations':MAX_ITER,
    'controller_objective':recipe['controller_objective'],'jackknife':recipe['jackknife'],
    'validation_grid':recipe['validation_grid'],'validation_guards':recipe['validation_guards']}
assert manifest13['plan']['recipe']==recipe_subset,'The scientific recipe changed after Notebook 13.'
prep_inventory_path=PREPARED/'artifact_inventory.json'
assert file_sha256(prep_inventory_path)==recipe['prepared_inventory_sha256']
prep_inventory={r['filename']:r for r in json.loads(prep_inventory_path.read_text())}
completion=json.loads((PREPARED/'notebook01_complete.json').read_text())
assert completion['ready_for_training'] and completion['artifact_inventory_sha256']==file_sha256(prep_inventory_path)
assert completion['protocol_sha256']==prep_inventory['protocol.json']['sha256']


def prepared_file(name):
    return verified_copy(PREPARED/name,'prepared/'+name,prep_inventory[name]['sha256'])


protocol=json.loads(prepared_file('protocol.json').read_text())
assert protocol['development_fault_types']==['dropout','bias','gain']
assert protocol['model_seeds']==SEEDS
RESULTS9={identity(r):r for r in report9['completed_results']}
RESULTS13={identity(r):r for r in report13['imported_results']+report13['test_results']}
assert set(RESULTS13)=={identity(j) for j in BASE_JOBS}
assert all(r['seed']==2026 for r in RESULTS13.values())
assert all(c['full_fold_coverage'] for c in report13['coverage'])
CACHE_INPUTS={identity(j):manifest9['plan']['cache_inputs'][identity(j)] for j in JOBS}
assert all(identity(j) in RESULTS9 for j in JOBS)
inventory13={r['filename']:r for r in json.loads((SOURCE13/'notebook13_artifact_inventory.json').read_text())}
IMPORTED_FILES={name:inventory13['notebook13_'+name+'.csv']['sha256']
    for name in ['aggregate_summary','test_summary','per_condition','per_participant']}
NEW_AUXILIARY_PAIRS=sum(4 if j['dataset']=='dsads' else 5 for j in JOBS)
NEW_CPU_HEADS=15*len(JOBS)
assert NEW_AUXILIARY_PAIRS==42 and NEW_CPU_HEADS==150 and len(JOBS)==10
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'new_jobs':JOBS,'imported_jobs':BASE_JOBS,
 'source_report_hashes':report_hashes,'source_manifest_hashes':manifest_hashes,
 'prepared_inventory_sha256':file_sha256(prep_inventory_path),'cache_inputs':CACHE_INPUTS,
 'imported13_csv_hashes':IMPORTED_FILES,'recipe':recipe_subset,
 'primary_method':PRIMARY,'methods':METHODS,'scopes':SCOPES,'seeds':SEEDS,
 'participant_separation':'source-only auxiliary models, normalization and controllers fitted separately per outer fold/seed; no fold transfer',
 'phase_order':'generate all ten source caches; fit/select all ten validation configurations; freeze all; evaluate saved test candidates',
 'aggregation':'condition means within fold, equal required-fold means within seed, equal COMPLETE-seed means; incomplete seeds excluded from overall aggregates',
 'seed_interpretation':'same participants and outer splits; original cached signal/corruption seeds and auxiliary seeds vary; controller head randomization and source-jackknife partition seed fixed at 2026',
 'test_status':'exploratory after previous test inspection; stuck excluded from fitting, earlier test outcomes inspected',
 'jackknife_limitation':report11['jackknife_limitation'],
 'confidence_intervals_or_p_values':False,'new_auxiliary_pairs_required':NEW_AUXILIARY_PAIRS,'new_cpu_heads_required':NEW_CPU_HEADS,
 'software':{'auxiliary_torch_version':recipe['software']['torch'],'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__}}
STUDY_SIGNATURE=json_signature(plan)
path=OUTPUT/'notebook14_study_manifest.json'
if path.exists():
    assert json.loads(path.read_text())['signature']==STUDY_SIGNATURE,(
        'Study inputs, code, recipe or environment changed. Keep the original settings and input outputs.')
else:save_json(path,{'signature':STUDY_SIGNATURE,'plan':plan})
print('Three-seed replication plan frozen:',STUDY_SIGNATURE[:16],flush=True)


def source_run_file(stage,job,name):
    assert stage==9
    result=RESULTS9[identity(job)]
    record=next(a for a in result['artifacts'] if a['filename']==name)
    relative='runs/'+identity(job)+'/'+name
    return verified_copy(SOURCE9/relative,'nb09/'+relative,record['sha256'])


def imported_table(name):
    filename='notebook13_'+name+'.csv'
    return pd.read_csv(verified_copy(SOURCE13/filename,'nb13/'+filename,IMPORTED_FILES[name]))


def historical_epoch_estimate():
    total=0.
    try:
        for dataset in ['dsads','har']:
            pilot=next(r for r in report11['completed_results'] if r['dataset']==dataset)
            seconds=0.
            for b in range(pilot['auxiliary_pairs']):
                for kind in ['classifier','reconstructor']:
                    p=SOURCE11/'runs'/identity(pilot)/'auxiliary'/f'cohort_{b:02d}'/(kind+'_history.csv')
                    seconds+=float(pd.read_csv(p)['seconds'].sum())
            jobs=sum(j['dataset']==dataset for j in JOBS)
            total+=seconds*jobs
        minutes=total/60.
        print(f'Historical pilot estimate for 42 new auxiliary pairs: {minutes:.1f} minutes of epoch work; excludes inference, CPU heads and I/O.',flush=True)
        print('This is a rough historical estimate; actual Colab runtime can differ.',flush=True)
        return minutes
    except (OSError,KeyError,StopIteration,ValueError):
        print('Historical epoch estimate unavailable. All new fits still resume normally.',flush=True)
        return None


def saved_phase_results(name):
    result=[]
    for job in JOBS:
        run=OUTPUT/'runs'/identity(job);p=run/name
        if p.exists():
            item=json.loads(p.read_text());assert_saved_artifacts(run,item)
            assert all(item[k]==job[k] for k in ['dataset','fold','seed'])
            result.append(item)
    return result


def collect_results(new_results):
    summary=imported_table('test_summary')
    cases=imported_table('per_condition');people=imported_table('per_participant')
    assert set(summary.method)==set(METHODS) and set(summary.seed)=={2026}
    assert set(summary[['dataset','fold','seed']].drop_duplicates().apply(lambda r:identity(r),axis=1))==set(RESULTS13)
    original_aggregate=imported_table('aggregate_summary')
    computed=summary.groupby(['dataset','method','scope'],sort=False)[METRICS+['sensor_action_intervention_rate']].mean().reset_index()
    assert_table_parity(computed,original_aggregate,['dataset','method','scope'],METRICS+['sensor_action_intervention_rate'])
    all_results=list(RESULTS13.values())+new_results
    summaries=[summary];case_frames=[cases];person_frames=[people]
    for result in new_results:
        run=OUTPUT/'runs'/identity(result)
        summaries.append(pd.read_csv(run/'test_summary.csv'))
        case_frames.append(pd.read_csv(run/'per_condition.csv'))
        person_frames.append(pd.read_csv(run/'per_participant.csv'))
        base=RESULTS13[identity({**result,'seed':2026})]
        assert result['test_participants']==base['test_participants'] and result['test_windows']==base['test_windows']
    summary=pd.concat(summaries,ignore_index=True)
    cases=pd.concat(case_frames,ignore_index=True);people=pd.concat(person_frames,ignore_index=True)
    assert not summary.duplicated(['dataset','fold','seed','method','scope']).any()
    assert not cases.duplicated(['dataset','fold','seed','method','condition']).any()
    assert not people.duplicated(['dataset','fold','seed','method','scope','subject']).any()
    for job in all_results:
        subset=summary[(summary.dataset==job['dataset'])&(summary.fold==job['fold'])&(summary.seed==job['seed'])]
        assert len(subset)==len(METHODS)*len(SCOPES) and set(subset.method)==set(METHODS)
    coverage=[]
    for dataset in ['dsads','har']:
        required=4 if dataset=='dsads' else 1
        for seed in SEEDS:
            current=[r for r in all_results if r['dataset']==dataset and r['seed']==seed]
            assert len({r['fold'] for r in current})==len(current)<=required
            seen=set()
            for result in current:
                p=set(result['test_participants']);assert not seen&p,'Participant appears in multiple test folds within a seed.'
                seen|=p
            if len(current)==required:assert len(seen)==(8 if dataset=='dsads' else 9)
            coverage.append({'dataset':dataset,'seed':seed,'completed_folds':len(current),'required_folds':required,
              'full_fold_coverage':len(current)==required,'unique_test_participants':len(seen),
              'unique_test_windows':sum(r['test_windows'] for r in current)})
    coverage=pd.DataFrame(coverage)
    cols=METRICS+['sensor_action_intervention_rate']
    per_seed=summary.groupby(['dataset','seed','method','scope'],sort=False)[cols].mean().reset_index()
    per_seed=per_seed.merge(coverage,on=['dataset','seed'],validate='many_to_one')
    complete=per_seed[per_seed.full_fold_coverage].copy()
    aggregate=complete.groupby(['dataset','method','scope'],sort=False)[cols].mean().reset_index()
    spread=complete.groupby(['dataset','method','scope'],sort=False)[METRICS].std(ddof=1).reset_index()
    spread=spread.rename(columns={m:m+'_seed_std' for m in METRICS})
    aggregate=aggregate.merge(spread,on=['dataset','method','scope'],validate='one_to_one')
    dataset_coverage=[]
    for dataset,group in coverage.groupby('dataset',sort=False):
        full=group[group.full_fold_coverage]
        assert len(full)>=1,'The imported seed must be complete.'
        assert full.unique_test_participants.nunique()==full.unique_test_windows.nunique()==1
        dataset_coverage.append({'dataset':dataset,'complete_seeds':len(full),'required_seeds':len(SEEDS),
          'full_seed_coverage':len(full)==len(SEEDS),'complete_configurations':int(full.completed_folds.sum()),
          'required_configurations':int(group.required_folds.sum()),
          'unique_test_participants':int(full.unique_test_participants.iloc[0]),
          'unique_test_windows':int(full.unique_test_windows.iloc[0]),
          'included_seeds':json.dumps(full.seed.astype(int).tolist())})
    dataset_coverage=pd.DataFrame(dataset_coverage)
    aggregate=aggregate.merge(dataset_coverage,on='dataset',validate='many_to_one')
    # Comparisons preserve paired fold/seed scores, never fuse inputs from different corruptions.
    comparison,harm=build_audits(summary,cases,people)
    seed_rows=[]
    for (dataset,seed,method,comparator,scope),group in comparison.groupby(['dataset','seed','method','comparator','scope'],sort=False):
        cov=coverage[(coverage.dataset==dataset)&(coverage.seed==seed)].iloc[0]
        seed_rows.append({'dataset':dataset,'seed':int(seed),'method':method,'comparator':comparator,'scope':scope,
          'primary_method':method==PRIMARY,'completed_folds':int(cov.completed_folds),'required_folds':int(cov.required_folds),
          'full_fold_coverage':bool(cov.full_fold_coverage),
          **{k:float(group[k].mean()) for k in ['policy_macro_f1_pct','comparator_macro_f1_pct','delta_macro_f1_pp',
                 'policy_harmful_rate_pct','comparator_harmful_rate_pct']},
          'positive_folds':int((group.delta_macro_f1_pp>1e-12).sum()),
          'negative_folds':int((group.delta_macro_f1_pp< -1e-12).sum()),
          'tied_folds':int((group.delta_macro_f1_pp.abs()<=1e-12).sum())})
    per_seed_comparison=pd.DataFrame(seed_rows)
    grouped=[]
    for (dataset,method,comparator,scope),group in per_seed_comparison[per_seed_comparison.full_fold_coverage].groupby(
            ['dataset','method','comparator','scope'],sort=False):
        eligible=set(group.seed.tolist())
        runs=comparison[(comparison.dataset==dataset)&comparison.seed.isin(eligible)&(comparison.method==method)
                        &(comparison.comparator==comparator)&(comparison.scope==scope)]
        cov=dataset_coverage[dataset_coverage.dataset==dataset].iloc[0]
        grouped.append({'dataset':dataset,'method':method,'comparator':comparator,'scope':scope,'primary_method':method==PRIMARY,
          'complete_seeds':len(group),'required_seeds':len(SEEDS),'full_seed_coverage':len(group)==len(SEEDS),
          'configurations':len(runs),'unique_test_participants':int(cov.unique_test_participants),
          **{k:float(group[k].mean()) for k in ['policy_macro_f1_pct','comparator_macro_f1_pct','delta_macro_f1_pp',
                 'policy_harmful_rate_pct','comparator_harmful_rate_pct']},
          'delta_seed_std_pp':float(group.delta_macro_f1_pp.std(ddof=1)),
          'positive_seeds':int((group.delta_macro_f1_pp>1e-12).sum()),
          'negative_seeds':int((group.delta_macro_f1_pp< -1e-12).sum()),
          'tied_seeds':int((group.delta_macro_f1_pp.abs()<=1e-12).sum()),
          'positive_runs':int((runs.delta_macro_f1_pp>1e-12).sum()),
          'negative_runs':int((runs.delta_macro_f1_pp< -1e-12).sum()),
          'tied_runs':int((runs.delta_macro_f1_pp.abs()<=1e-12).sum())})
    comparisons=pd.DataFrame(grouped)
    diagnostics=[]
    for row in aggregate[aggregate.method==PRIMARY].to_dict('records'):
        included=json.loads(row['included_seeds'])
        audit=harm[(harm.dataset==row['dataset'])&(harm.method==PRIMARY)&(harm.scope==row['scope'])&harm.seed.isin(included)]
        diagnostics.append({'dataset':row['dataset'],'scope':row['scope'],'complete_seeds':int(row['complete_seeds']),
          'macro_f1_pct':row['macro_f1']*100,'mean_harm_pct':row['harmful_rate']*100,
          'runs':len(audit),'runs_exceeding_1pct_mean_harm':int((~audit.mean_harm_le_1pct).sum()),
          'maximum_condition_harm_pct':float(audit.maximum_condition_harm_pct.max()),
          'maximum_participant_scope_harm_pct':float(audit.maximum_participant_mean_harm_pct.max()),
          'descriptive_only':True})
    diagnostics=pd.DataFrame(diagnostics)
    for name,frame in [('aggregate_summary',aggregate),('per_seed_summary',per_seed),('test_summary',summary),
      ('per_condition',cases),('per_participant',people),('comparisons',comparisons),
      ('per_seed_comparisons',per_seed_comparison),('per_fold_comparisons',comparison),('harm_audit',harm),
      ('coverage',coverage),('dataset_coverage',dataset_coverage),('primary_diagnostics',diagnostics)]:
        save_csv(OUTPUT/f'notebook14_{name}.csv',frame)
    return aggregate,comparisons,coverage,dataset_coverage,diagnostics


estimate=historical_epoch_estimate();paused=None
try:
    source_results=[]
    if RUN_MODE=='all' or DEVICE.type=='cuda':
        for job in JOBS:source_results.append(generate_sources(job))
    else:
        for job in JOBS:
            r=source_ready(job)
            if r is None:raise PauseStudy('Auxiliary caches are incomplete. Select T4 GPU and rerun this same cell first.')
            source_results.append(r)
    if RUN_MODE=='auto' and DEVICE.type=='cuda' and not all((OUTPUT/'runs'/identity(j)/'validation_complete.json').exists() for j in JOBS):
        raise PauseStudy('GPU STAGE COMPLETE. Change runtime type to CPU, then rerun this same cell for controllers and cached test evaluation.')
    validation_results=[]
    for job,ready in zip(JOBS,source_results):validation_results.append(fit_policies(job,ready))
    freeze={'study_signature':STUDY_SIGNATURE,'primary_method':PRIMARY,'methods':METHODS,
      'validation_complete':{identity(j):file_sha256(OUTPUT/'runs'/identity(j)/'validation_complete.json') for j in JOBS},
      'frozen_policy_hashes':{identity(j):file_sha256(OUTPUT/'runs'/identity(j)/'frozen_policies.json') for j in JOBS}}
    frozen=OUTPUT/'notebook14_frozen_test_plan.json'
    if frozen.exists():assert json.loads(frozen.read_text())==freeze,'Frozen validation choices changed.'
    else:save_json(frozen,freeze)
    print('\nAll ten new configurations frozen before new test-array access:',json_signature(freeze)[:16],flush=True)
    for job,result in zip(JOBS,validation_results):phase_b(job,result)
except PauseStudy as e:
    paused=str(e);print('\nPAUSED:',paused,flush=True)

# Recover saved phase records after every pause. A GPU rerun cannot erase CPU progress.
source_results=saved_phase_results('source_ready.json')
validation_results=saved_phase_results('validation_complete.json')
test_results=saved_phase_results('test_complete.json')
aggregate,comparisons,coverage,dataset_coverage,diagnostics=collect_results(test_results)
complete=len(test_results)==len(JOBS)
if complete:paused=None
report={'code_version':CODE_VERSION,'study_signature':STUDY_SIGNATURE,'study_complete':complete,
 'new_source_configurations_complete':len(source_results),'new_validation_configurations_complete':len(validation_results),
 'new_test_configurations_complete':len(test_results),'new_configurations_required':len(JOBS),
 'total_test_configurations_complete':len(BASE_JOBS)+len(test_results),'total_configurations_required':len(BASE_JOBS)+len(JOBS),
 'source_results':source_results,'validation_results':validation_results,'test_results':test_results,
 'imported_results':list(RESULTS13.values()),'coverage':coverage.to_dict('records'),
 'dataset_coverage':dataset_coverage.to_dict('records'),'primary_diagnostics':diagnostics.to_dict('records'),
 'pause_reason':paused,'runtime_mode':RUN_MODE,'primary_method':PRIMARY,'methods_reported':len(METHODS),
 'new_auxiliary_pairs_required':NEW_AUXILIARY_PAIRS,'new_cpu_heads_required':NEW_CPU_HEADS,
 'original_signal_models_retrained':0,'new_baseline_signal_inference':0,'previous_outputs_modified':False,
 'seed_2026_notebook13_metrics_reproduced':True,'new_test_decisions':bool(test_results),'test_based_threshold_selection':False,
 'per_configuration_settings_selected_on':'own validation participants only; same frozen grid and empirical guards',
 'historical_auxiliary_epoch_minutes_estimate':estimate,'test_status':plan['test_status'],
 'aggregation':plan['aggregation'],'seed_interpretation':plan['seed_interpretation'],
 'seed_standard_deviation_is_confidence_interval':False,'confidence_intervals_or_p_values':False,
 'harm_or_novelty_guarantee':False,'jackknife_limitation':plan['jackknife_limitation'],
 'runtime_torch_version':str(torch.__version__)}
save_json(OUTPUT/'notebook14_run_report.json',report)
if validation_results:
    val=pd.concat([pd.read_csv(OUTPUT/'runs'/identity(r)/'validation_summary.csv') for r in validation_results],ignore_index=True)
    save_csv(OUTPUT/'notebook14_validation_summary.csv',val)
if source_results:
    audit=[]
    for r in source_results:
        for a in r['source_audit']:
            row=dict(a)
            for k in ['training_participants','held_participants']:row[k]=json.dumps(row[k])
            audit.append(row)
    save_csv(OUTPUT/'notebook14_source_audit.csv',pd.DataFrame(audit))
if validation_results:
    audit=[]
    for r in validation_results:
        for a in r['matched_control_audit']:
            row=dict(a)
            for k in ['source_participants','selected_participants','selected_counts_by_participant']:row[k]=json.dumps(row[k])
            audit.append(row)
    save_csv(OUTPUT/'notebook14_matched_control_audit.csv',pd.DataFrame(audit))
print('\nFOLD AND SEED COVERAGE | same participants across seeds',flush=True)
print(coverage.to_string(index=False),flush=True)
view=comparisons[(comparisons.method==PRIMARY)&comparisons.comparator.isin(
    ['keep','harm_aware_policy','oof_accuracy_utility','oof_f1_utility','window_mean_f1','residual_stacking_safe'])]
print('\nPRIMARY EXPLORATORY COMPARISONS | complete seeds only | percentages; deltas in pp',flush=True)
print(view[['dataset','scope','comparator','complete_seeds','required_seeds','policy_macro_f1_pct',
    'comparator_macro_f1_pct','delta_macro_f1_pp','positive_runs','negative_runs','policy_harmful_rate_pct']].round(3).to_string(index=False),flush=True)
per_seed=pd.read_csv(OUTPUT/'notebook14_per_seed_summary.csv')
view=per_seed[per_seed.method==PRIMARY].copy()
view['macro_f1_pct']=view.macro_f1*100;view['harmful_rate_pct']=view.harmful_rate*100
print('\nPRIMARY PER-SEED SCORES | incomplete folds explicitly labeled',flush=True)
print(view[['dataset','seed','scope','completed_folds','required_folds','full_fold_coverage','macro_f1_pct','harmful_rate_pct']].round(3).to_string(index=False),flush=True)
print('\nOBSERVED PRIMARY HARM | empirical diagnostics, not guarantees',flush=True)
print(diagnostics.round(3).to_string(index=False),flush=True)
print('\nNOTEBOOK 14','COMPLETE' if complete else 'PAUSED',flush=True)
print(f'New source configurations: {len(source_results)}/10 | New validation configurations: {len(validation_results)}/10 | New test configurations: {len(test_results)}/10',flush=True)
print(f'Total configurations including imported seed 2026: {len(BASE_JOBS)+len(test_results)}/15 | Methods: {len(METHODS)}',flush=True)
print('Seed 2026 reproduced. Original main models and earlier outputs unchanged. No significance or risk guarantees.',flush=True)
print('Outputs:',OUTPUT,flush=True)
if not complete:print('Next action:',paused,flush=True)
print('Share final output, notebook14_aggregate_summary.csv, notebook14_comparisons.csv, notebook14_per_seed_summary.csv, notebook14_harm_audit.csv, notebook14_run_report.json.',flush=True)
records=[{'filename':p.name,'sha256':file_sha256(p),'bytes':p.stat().st_size}
  for p in sorted(OUTPUT.iterdir()) if p.is_file() and p.name!='notebook14_artifact_inventory.json']
save_json(OUTPUT/'notebook14_artifact_inventory.json',records)


Mounted at /content/drive
NOTEBOOK 14 | frozen-method three-seed replication | one executable cell
Imported: all five seed-2026 configurations. New: ten configurations, seeds 2027 and 2028.
Device: CPU | mode: auto
42 auxiliary classifier/reconstructor pairs and 150 CPU heads; existing main signal models/caches reused.
Earlier test outcomes were inspected. All comparisons remain exploratory.
Three-seed replication plan frozen: ab525f4f83584db9
Historical pilot estimate for 42 new auxiliary pairs: 20.6 minutes of epoch work; excludes inference, CPU heads and I/O.
This is a rough historical estimate; actual Colab runtime can differ.
  Reused frozen validation: dsads_fold_0_seed_2027
  Reused frozen validation: dsads_fold_1_seed_2027
  Reused frozen validation: dsads_fold_2_seed_2027
  Reused frozen validation: dsads_fold_3_seed_2027
  Reused frozen validation: har_fold_0_seed_2027
  Reused frozen validation: dsads_fold_0_seed_2028
  Reused frozen validation: dsads_fold_1_seed_2028
  Reus